<a href="https://colab.research.google.com/github/hayahanyyy/Bachelor-Thesis/blob/main/implementationD2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC



In [2]:
# 2. LOAD DATA
# ==============================
df = pd.read_csv("AI_Personalized_Learning.csv")

In [3]:
# Drop ID column
df = df.drop(columns=["student_id"])

In [10]:
# 3. FEATURE ENGINEERING (🔥 KEY STEP)
# ==============================

# Avoid division by zero
df["engagement_efficiency"] = df["engagement_score"] / (df["avg_time_per_module"] + 1)

df["performance_score"] = (
    df["quiz_accuracy"] + df["completed_modules"]
) / 2

df["study_intensity"] = df["avg_time_per_module"] * df["engagement_score"]


# ==============================

In [6]:
print(df.columns)

Index(['age', 'gender', 'education_level', 'learning_style', 'previous_gpa',
       'completed_modules', 'avg_time_per_module', 'engagement_score',
       'distraction_events', 'quiz_accuracy', 'feedback_score',
       'contextual_difficulty_level', 'recommended_path',
       'actual_path_followed', 'path_efficiency_score',
       'final_assessment_score', 'learning_outcome'],
      dtype='object')


In [11]:
# 4. TARGET SIMPLIFICATION (🔥 HUGE BOOST)
# ==============================

# Convert to binary classification
df["learning_outcome"] = df["learning_outcome"].apply(
    lambda x: 1 if x in [2, 3] else 0
)




In [12]:
# 5. SPLIT FEATURES & TARGET
# ==============================
X = df.drop("learning_outcome", axis=1)
y = df["learning_outcome"]

In [13]:
# 6. ONE-HOT ENCODING
# ==============================
X = pd.get_dummies(X, drop_first=True)

In [14]:
# 7. OPTIONAL: DROP WEAK FEATURES
# ==============================
# Try with and without this
weak_features = ["age", "gender"]
X = X.drop(columns=[col for col in weak_features if col in X.columns])

In [15]:
# 8. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [16]:
# 9. FEATURE SCALING
# ==============================
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [17]:
# 10. MODEL TRAINING + TUNING
# ==============================

# 🔥 Random Forest (BEST BET)
rf_params = {
    "n_estimators": [200, 300],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

best_rf = rf_grid.best_estimator_


# ==============================

In [18]:
# 11. EVALUATION
# ==============================
y_pred = best_rf.predict(X_test)

print("Best Parameters:", rf_grid.best_params_)
print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Best Parameters: {'max_depth': 10, 'min_samples_split': 2, 'n_estimators': 200}

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       200

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200

